In [ ]:
import json, os

import sys
sys.path.append("..")
import src.config as config

In [13]:
out_dir = f"{config.output_folder}/{config.d_i}-{config.d_f}/results"


traces = {}
for file in os.listdir(out_dir):
    assert file.endswith(".json")
    with open(os.path.join(out_dir, file), 'r') as f:
        method = file.split(".")[0]
        traces[method] = json.load(f)["traces"]



In [14]:
carbion_footprints = {}
water_footprints = {}
land_use_footprints = {}

# Iterate over each method in traces
for method, job_traces in traces.items():
    # Aggregate carbon footprint over time
    carbion_footprints[method] = [.0]*len(config.timestamps)
    water_footprints[method] = [.0]*len(config.timestamps)
    land_use_footprints[method] = [.0]*len(config.timestamps)
    for trace in job_traces:
        for t, _ in enumerate(config.timestamps):
            carbion_footprints[method][t] += trace["carbon_intensity"][t]*trace["energy_consumption"][t]
            water_footprints[method][t] += trace["water_intensity"][t]*trace["energy_consumption"][t]
            land_use_footprints[method][t] += trace["land_use_intensity"][t]*trace["energy_consumption"][t]



In [15]:
def plot(footprints, factor):
    import matplotlib.pyplot as plt

    # Plot the data
    plt.figure(figsize=(10, 6))
    for method, footprint in footprints.items():
        # Plot the aggregated carbon footprint over time
        plt.plot(config.timestamps, footprint, label=f"{method}")
    plt.xlabel("Time")
    plt.ylabel(f"Aggregated {factor} Footprint")
    plt.title(f"Aggregated {factor} Footprint Over Time")
    plt.legend()
    plt.grid()
    plt.xticks(rotation=45)
    plt.tight_layout()
    # Save the plot
    plot_dir = f"{config.output_folder}/{config.d_i}-{config.d_f}/plots"
    if not os.path.exists(plot_dir):
        os.makedirs(plot_dir)
    plot_path = os.path.join(plot_dir, f"{factor.lower()}_footprint.png")
    plt.savefig(plot_path)
    plt.close()


In [16]:
plot(carbion_footprints, "Carbon")
plot(water_footprints, "Water")

plot(land_use_footprints, "Land use")
